# Public repository note

This notebook is an output-cleared code copy. It requires locally authorised data and is not runnable from the public repository alone. Green Street raw data, intermediate files, derived aggregates and outputs are not distributed.


# 09 Office-Stock Adjustment and Retail Change

This notebook answers the second dissertation research question:

> How are different forms of post-pandemic office-stock adjustment,
> particularly the fragmentation of office units and the persistence of
> large-office structures, associated with local retail change?

Two analytical expectations guide the analysis. First, fragmentation may
represent adaptation to changing occupier demand and may therefore accompany
more resilient retail outcomes. Second, areas retaining a stable concentration
of large office units may experience weaker retail renewal.

The main analysis uses MSOA-year observations in the five predefined London
office submarkets. Office restructuring is measured from OpenLocal
hereditaments. Retail outcomes are kept as two complementary families:

1. **OpenLocal commercial-property stock:** retail units, rateable value,
floorspace and an occupation-based vacancy proxy derived from official VOA
occupation-state records.
2. **Green Street consumer-facing retail activity:** active premises, vacancy,
   long-term vacancy, turnover and net formation.

The primary predictor is a transparent fragmentation index. A higher value
means that, relative to 2019, an office area contains more units, a smaller
median unit, a larger share of units below 100 square metres, and a smaller
share at or above 1,000 square metres. Total office floorspace change is
controlled separately so that internal restructuring is not confused with
simple expansion or contraction.

In [ ]:
from pathlib import Path
import os
import gc
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
import statsmodels.formula.api as smf
from matplotlib.colors import Normalize, TwoSlopeNorm
from patsy import build_design_matrices
from shapely import from_wkb

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 180)

BASE = Path(os.environ.get("DISSERTATION_WORKSPACE", Path.cwd().resolve()))
OUTPUT_DIR = BASE / "outputs" / "restricted_h2_office_restructuring"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OPENLOCAL_PATH = BASE / "2025-12-31-shuting-yang-greenstreet-retail.parquet"
MSOA_PATH = (
    BASE / "outputs" / "restricted_msoa_origin_exposure_analysis"
    / "london_msoa_2021_boundaries.geojson"
)
OFFICE_MARKETS_PATH = BASE / "London_Office_Markets_V1.geojson"
LAD_PATH = BASE / "Local_Authority_Districts_May_2024_Boundaries_UK_BGC_3503156029110784919.geojson"
GS_POST_PANEL_PATH = (
    BASE / "outputs" / "restricted_greenstreet_historical_poi_integration"
    / "greenstreet_historical_poi_msoa_post_panel.csv"
)
H1_STATION_METRICS_PATH = (
    BASE / "outputs" / "restricted_h1_fine_grained_analysis"
    / "tfl_all_station_commuter_shock.csv"
)

YEARS = [2019, 2023, 2024, 2025]
POST_YEARS = [2023, 2024, 2025]
SMALL_OFFICE_SQM = 100
LARGE_OFFICE_SQM = 1000
MIN_BASELINE_OFFICE_UNITS = 20
MIN_BASELINE_RETAIL_UNITS = 20
MIN_BASELINE_GS_ACTIVE = 20

sns.set_theme(style="whitegrid", context="talk")
PALETTE = {
    "orange": "#d97732",
    "orange_dark": "#9f3b12",
    "blue": "#3f7cac",
    "blue_dark": "#174a74",
    "green": "#3f806b",
    "red": "#b64f4f",
    "grey": "#667085",
    "light": "#f2f3f2",
}

## 1. Spatial framework

The five office submarkets are fixed before outcome analysis. Individual
OpenLocal office points are assigned first to an office-market polygon and
then to an MSOA. Where an MSOA contains office records from more than one
submarket, it is assigned to the submarket accounting for the largest
baseline office floorspace. This preserves an MSOA analysis unit while using
the commercial market boundaries to define the office sample.

In [ ]:
submarket_crosswalk = {
    "Mayfair": "West End",
    "Soho": "West End",
    "St James's": "West End",
    "Covent Garden": "West End",
    "Fitzrovia": "West End",
    "North of Oxford Street": "West End",
    "Paddington": "West End",
    "Knightsbridge": "West End",
    "Victoria": "West End",
    "City Core": "City",
    "Midtown": "Tech Belt & Midtown",
    "Bloomsbury": "Tech Belt & Midtown",
    "Clerkenwell": "Tech Belt & Midtown",
    "Euston": "Tech Belt & Midtown",
    "Kings Cross": "Tech Belt & Midtown",
    "Shoreditch": "Tech Belt & Midtown",
    "Camden": "Tech Belt & Midtown",
    "Aldgate & Whitechapel": "Tech Belt & Midtown",
    "Canary Wharf": "Canary Wharf",
    "Southbank": "Southbank",
    "Waterloo": "Southbank",
    "Vauxhall, Nine Elms and Battersea": "Southbank",
}

msoa = gpd.read_file(MSOA_PATH).to_crs("EPSG:4326")
msoa = msoa[["MSOA21CD", "MSOA21NM", "geometry"]].drop_duplicates("MSOA21CD")
office_markets = gpd.read_file(OFFICE_MARKETS_PATH).to_crs("EPSG:4326")
office_markets["study_submarket"] = office_markets["Market"].map(submarket_crosswalk)
core_markets = office_markets.dropna(subset=["study_submarket"]).copy()
lad = gpd.read_file(LAD_PATH).to_crs("EPSG:4326")
london_lad = lad[lad["LAD24CD"].astype(str).str.startswith("E09")].copy()

print("London MSOAs:", len(msoa))
print("Core office-market polygons:", len(core_markets))
print("Study submarkets:", sorted(core_markets["study_submarket"].unique()))

## 2. OpenLocal office hereditaments

OpenLocal is quarterly. To avoid treating four observations of the same
property as four different offices, records are deduplicated by quarter and
UARN. Indicators are calculated separately for every quarter and then averaged
to annual values. The 2019 annual average is the pre-pandemic baseline.

In [ ]:
def scan_openlocal_points(category, target_geography, target_columns, batch_size=200_000):
    columns = [
        "period", "uarn", "occupation_state", "total_floor_area",
        "rateable_value", "geometry", "category_group",
    ]
    parquet = pq.ParquetFile(OPENLOCAL_PATH)
    selected = []
    minx, miny, maxx, maxy = target_geography.total_bounds
    for batch in parquet.iter_batches(columns=columns, batch_size=batch_size):
        frame = batch.to_pandas()
        period_text = frame["period"].astype("string")
        keep_year = (
            period_text.str.startswith("2019")
            | period_text.str.startswith("2023")
            | period_text.str.startswith("2024")
            | period_text.str.startswith("2025")
        )
        frame = frame[
            frame["category_group"].eq(category) & keep_year
        ].drop(columns=["category_group"])
        if frame.empty:
            continue
        geometry = from_wkb(frame["geometry"].map(bytes.fromhex).to_numpy())
        points = gpd.GeoDataFrame(
            frame.drop(columns=["geometry"]),
            geometry=geometry,
            crs="EPSG:4326",
        )
        points = points.cx[minx:maxx, miny:maxy]
        if points.empty:
            continue
        joined = gpd.sjoin(
            points,
            target_geography[target_columns + ["geometry"]],
            how="inner",
            predicate="within",
        ).drop(columns=["index_right"])
        selected.append(joined)
        del frame, geometry, points, joined
        gc.collect()
    if not selected:
        return gpd.GeoDataFrame(columns=columns, geometry=[], crs="EPSG:4326")
    result = pd.concat(selected, ignore_index=True)
    return gpd.GeoDataFrame(result, geometry="geometry", crs="EPSG:4326")

office_core = scan_openlocal_points(
    "OFFICE", core_markets, ["study_submarket"]
)
office_core["period"] = pd.to_datetime(office_core["period"])
office_core["year"] = office_core["period"].dt.year
office_core["total_floor_area"] = pd.to_numeric(
    office_core["total_floor_area"], errors="coerce"
)
office_core["rateable_value"] = pd.to_numeric(
    office_core["rateable_value"], errors="coerce"
)
office_core = gpd.sjoin(
    office_core,
    msoa,
    how="inner",
    predicate="within",
).drop(columns=["index_right"])
office_core = office_core.sort_values("period").drop_duplicates(
    ["period", "uarn"], keep="last"
)

print("Rows inside five office submarkets:", f"{len(office_core):,}")
print("Distinct office hereditaments:", f"{office_core['uarn'].nunique():,}")
print("MSOAs containing office records:", office_core["MSOA21CD"].nunique())
display(
    office_core.groupby("study_submarket")
    .agg(rows=("uarn", "size"), hereditaments=("uarn", "nunique"))
    .sort_values("hereditaments", ascending=False)
)

In [ ]:
def quarterly_office_metrics(group):
    size = pd.to_numeric(group["total_floor_area"], errors="coerce")
    valid = size.dropna()
    total_area = valid.sum(min_count=1)
    occupied = group["occupation_state"].astype("string").eq("OCCUPIED")
    vacant = group["occupation_state"].astype("string").eq("VACANT")
    classified = occupied | vacant
    return pd.Series({
        "office_units": group["uarn"].nunique(),
        "office_total_floor_area": total_area,
        "office_median_floor_area": valid.median(),
        "office_p25_floor_area": valid.quantile(0.25),
        "office_p75_floor_area": valid.quantile(0.75),
        "office_small_unit_share": (valid < SMALL_OFFICE_SQM).mean(),
        "office_large_unit_share": (valid >= LARGE_OFFICE_SQM).mean(),
        "office_large_floor_share": (
            valid[valid >= LARGE_OFFICE_SQM].sum() / total_area
            if pd.notna(total_area) and total_area > 0 else np.nan
        ),
        "office_total_rateable_value": pd.to_numeric(
            group["rateable_value"], errors="coerce"
        ).sum(min_count=1),
        "office_vacancy_proxy": (
            vacant.sum() / classified.sum() if classified.sum() else np.nan
        ),
    })

office_quarter = (
    office_core.groupby(
        ["period", "year", "MSOA21CD", "MSOA21NM", "study_submarket"],
        observed=True,
    )
    .apply(quarterly_office_metrics, include_groups=False)
    .reset_index()
)

office_annual = (
    office_quarter.groupby(
        ["year", "MSOA21CD", "MSOA21NM", "study_submarket"],
        observed=True,
    )
    .agg(
        n_quarters=("period", "nunique"),
        **{
            col: (col, "mean")
            for col in office_quarter.columns
            if col.startswith("office_")
        },
    )
    .reset_index()
)

# Dominant submarket is based on 2019 office floorspace.
dominant_submarket = (
    office_annual[office_annual["year"].eq(2019)]
    .sort_values("office_total_floor_area", ascending=False)
    .drop_duplicates("MSOA21CD")
    [["MSOA21CD", "study_submarket"]]
    .rename(columns={"study_submarket": "dominant_submarket"})
)
office_annual = office_annual.merge(
    dominant_submarket, on="MSOA21CD", how="left"
)
office_annual = office_annual[
    office_annual["study_submarket"].eq(office_annual["dominant_submarket"])
].copy()
office_annual["study_submarket"] = office_annual["dominant_submarket"]
office_annual = office_annual.drop(columns=["dominant_submarket"])

baseline_office = (
    office_annual[office_annual["year"].eq(2019)]
    .set_index("MSOA21CD")
)
eligible_codes = baseline_office.index[
    baseline_office["office_units"].ge(MIN_BASELINE_OFFICE_UNITS)
].tolist()
office_annual = office_annual[
    office_annual["MSOA21CD"].isin(eligible_codes)
    & office_annual["n_quarters"].ge(4)
].copy()

print("Eligible RQ2 MSOAs:", len(eligible_codes))
display(
    baseline_office.loc[eligible_codes]
    .groupby("study_submarket")
    .agg(
        MSOAs=("MSOA21NM", "nunique"),
        office_units=("office_units", "sum"),
        median_unit_sqm=("office_median_floor_area", "median"),
    )
)

## 3. Fragmentation index

Four changes relative to 2019 describe internal restructuring:

- growth in the number of office units;
- decline in median office-unit size;
- growth in the share of units below 100 square metres;
- decline in the share of units at or above 1,000 square metres.

Each component is standardised so that differently scaled fields contribute
comparably. Their mean forms the fragmentation index. Total office floorspace
change is excluded from the index and enters the model as a separate control.
This allows a high score to mean internal reconfiguration rather than simply a
larger or smaller office market.

The hypothesis also contains a contrasting condition: office areas that remain
dominated by large units. A persistent-large-office score is therefore
calculated as the baseline share of floorspace in units of at least 1,000
square metres minus the absolute subsequent change in that share. A high value
means that large units accounted for substantial floorspace in 2019 and that
this structure remained comparatively stable.

In [ ]:
office_base_cols = [
    "office_units", "office_total_floor_area", "office_median_floor_area",
    "office_small_unit_share", "office_large_unit_share",
    "office_large_floor_share", "office_total_rateable_value",
    "office_vacancy_proxy",
]
base = (
    office_annual[office_annual["year"].eq(2019)]
    [["MSOA21CD"] + office_base_cols]
    .rename(columns={c: f"{c}_2019" for c in office_base_cols})
)
h2_office_panel = office_annual[
    office_annual["year"].isin(POST_YEARS)
].merge(base, on="MSOA21CD", how="inner", validate="many_to_one")

def safe_log_ratio(current, baseline):
    return np.log(current.clip(lower=1e-6) / baseline.clip(lower=1e-6))

h2_office_panel["office_unit_log_change"] = safe_log_ratio(
    h2_office_panel["office_units"],
    h2_office_panel["office_units_2019"],
)
h2_office_panel["office_median_size_log_change"] = safe_log_ratio(
    h2_office_panel["office_median_floor_area"],
    h2_office_panel["office_median_floor_area_2019"],
)
h2_office_panel["office_floor_log_change"] = safe_log_ratio(
    h2_office_panel["office_total_floor_area"],
    h2_office_panel["office_total_floor_area_2019"],
)
h2_office_panel["office_value_log_change"] = safe_log_ratio(
    h2_office_panel["office_total_rateable_value"],
    h2_office_panel["office_total_rateable_value_2019"],
)
h2_office_panel["office_vacancy_proxy_change"] = (
    h2_office_panel["office_vacancy_proxy"]
    - h2_office_panel["office_vacancy_proxy_2019"]
)
h2_office_panel["small_unit_share_change"] = (
    h2_office_panel["office_small_unit_share"]
    - h2_office_panel["office_small_unit_share_2019"]
)
h2_office_panel["large_unit_share_change"] = (
    h2_office_panel["office_large_unit_share"]
    - h2_office_panel["office_large_unit_share_2019"]
)
h2_office_panel["large_floor_share_change"] = (
    h2_office_panel["office_large_floor_share"]
    - h2_office_panel["office_large_floor_share_2019"]
)
h2_office_panel["log_office_floor_area_2019"] = np.log1p(
    h2_office_panel["office_total_floor_area_2019"]
)

fragment_components = {
    "office_unit_log_change": 1,
    "office_median_size_log_change": -1,
    "small_unit_share_change": 1,
    "large_unit_share_change": -1,
}

def winsorised_z(series, lower=0.01, upper=0.99):
    x = pd.to_numeric(series, errors="coerce")
    lo, hi = x.quantile([lower, upper])
    clipped = x.clip(lo, hi)
    return (clipped - clipped.mean()) / clipped.std(ddof=0)

index_parts = []
for component, direction in fragment_components.items():
    z_col = f"z_{component}"
    h2_office_panel[z_col] = (
        direction * winsorised_z(h2_office_panel[component])
    )
    index_parts.append(z_col)

h2_office_panel["fragmentation_index_raw"] = h2_office_panel[
    index_parts
].mean(axis=1)
h2_office_panel["fragmentation_z"] = winsorised_z(
    h2_office_panel["fragmentation_index_raw"]
)
h2_office_panel["persistent_large_raw"] = (
    h2_office_panel["office_large_floor_share_2019"]
    - h2_office_panel["large_floor_share_change"].abs()
)
h2_office_panel["persistent_large_z"] = winsorised_z(
    h2_office_panel["persistent_large_raw"]
)

print("RQ2 office panel:", h2_office_panel.shape)
display(
    h2_office_panel[
        ["year", "MSOA21NM", "study_submarket", "fragmentation_z"]
        + list(fragment_components)
        + [
            "office_floor_log_change", "office_value_log_change",
            "office_vacancy_proxy_change", "persistent_large_z"
        ]
    ].head(10)
)

## 3.1 Office-market adjustment components

Before using the structural indices, this diagnostic keeps the underlying
office indicators separate. The left panel compares change in office-unit
count with change in median unit size: more units with smaller units is
consistent with subdivision, while fewer units with larger units is consistent
with consolidation. The right panel compares change in total floorspace with
change in total rateable value. In both panels, colour records the change in
the office occupation-based vacancy proxy.

The figure does **not** classify a vacancy decline as demand recovery. For
example, a move towards fewer, larger office units together with a lower
vacancy proxy can result from unit mergers as well as new demand. Its role is
to show why unit structure, floorspace, valuation and recorded occupation are
read together in RQ2.

In [ ]:
office_component_profile = (
    h2_office_panel.groupby(
        ["MSOA21CD", "MSOA21NM", "study_submarket"], as_index=False
    )
    [[
        "office_unit_log_change", "office_median_size_log_change",
        "office_floor_log_change", "office_value_log_change",
        "office_vacancy_proxy_change",
    ]]
    .mean()
)

# Express log changes as approximate percentage changes for the diagnostic.
for column in [
    "office_unit_log_change", "office_median_size_log_change",
    "office_floor_log_change", "office_value_log_change",
]:
    office_component_profile[f"{column}_pct"] = (
        100 * np.expm1(office_component_profile[column])
    )
office_component_profile["office_vacancy_pp_change"] = (
    100 * office_component_profile["office_vacancy_proxy_change"]
)

vacancy_bound = office_component_profile["office_vacancy_pp_change"].abs().quantile(0.98)
vacancy_bound = max(float(vacancy_bound), 0.5)
vacancy_norm = TwoSlopeNorm(vmin=-vacancy_bound, vcenter=0, vmax=vacancy_bound)

component_panels = [
    (
        "office_unit_log_change_pct", "office_median_size_log_change_pct",
        "A. Office-unit structure", "Office unit-count change from 2019 (%)",
        "Median office-unit-size change from 2019 (%)",
    ),
    (
        "office_floor_log_change_pct", "office_value_log_change_pct",
        "B. Office stock and valuation", "Total office-floorspace change from 2019 (%)",
        "Total office rateable-value change from 2019 (%)",
    ),
]

fig, axes = plt.subplots(1, 2, figsize=(13.2, 6.1), constrained_layout=True)
for position, (ax, (x_col, y_col, title, x_label, y_label)) in enumerate(zip(axes, component_panels)):
    plotted = office_component_profile.dropna(subset=[x_col, y_col, "office_vacancy_pp_change"])
    # Limit only the visual axes so a small number of tiny-baseline MSOAs do
    # not flatten the central distribution. Raw values remain in all models.
    x_lo, x_hi = plotted[x_col].quantile([0.02, 0.98])
    y_lo, y_hi = plotted[y_col].quantile([0.02, 0.98])
    plot_x = plotted[x_col].clip(x_lo, x_hi)
    plot_y = plotted[y_col].clip(y_lo, y_hi)
    scatter = ax.scatter(
        plot_x, plot_y,
        c=plotted["office_vacancy_pp_change"], cmap="RdBu_r", norm=vacancy_norm,
        s=52, alpha=0.84, edgecolor="#334155", linewidth=0.35,
    )
    ax.axhline(0, color="#64748b", linewidth=1.0, linestyle="--")
    ax.axvline(0, color="#64748b", linewidth=1.0, linestyle="--")
    ax.set_title(title, loc="left", fontsize=15, fontweight="bold")
    ax.set_xlabel(x_label, fontsize=11)
    ax.set_ylabel(y_label, fontsize=11)
    ax.set_xlim(x_lo - 0.06 * (x_hi - x_lo), x_hi + 0.06 * (x_hi - x_lo))
    ax.set_ylim(y_lo - 0.06 * (y_hi - y_lo), y_hi + 0.06 * (y_hi - y_lo))
    ax.grid(alpha=0.16)
    sns.despine(ax=ax)
    if position == 0:
        xlim, ylim = ax.get_xlim(), ax.get_ylim()
        ax.text(
            xlim[0] + 0.03 * (xlim[1] - xlim[0]),
            ylim[1] - 0.04 * (ylim[1] - ylim[0]),
            "Fewer, larger units\n(consolidation)",
            ha="left", va="top", fontsize=9.5, color="#475569",
        )
        ax.text(
            xlim[1] - 0.03 * (xlim[1] - xlim[0]),
            ylim[0] + 0.04 * (ylim[1] - ylim[0]),
            "More, smaller units\n(fragmentation)",
            ha="right", va="bottom", fontsize=9.5, color="#475569",
        )

cbar = fig.colorbar(scatter, ax=axes, shrink=0.82, pad=0.02)
cbar.set_label("Change in office occupation-based vacancy (percentage points)", fontsize=10.5)
fig.suptitle(
    "Objective 2 Office-Market Adjustment Components: 2023-2025 Mean Relative to 2019",
    x=0.01, ha="left", fontsize=20, fontweight="bold",
)
fig.savefig(
    FIGURE_DIR / "fig_00_h2_office_market_adjustment_components.png",
    dpi=300, bbox_inches="tight",
)
plt.show()

## 4. OpenLocal retail outcomes

Retail records are assigned to the eligible RQ2 MSOAs. Quarterly indicators are
again averaged to annual values. The main stock outcomes are expressed as log
changes from 2019; vacancy is an absolute change in the share of records
classified as vacant.

In [ ]:
# Release row-level office objects before reading the large retail extract.
office_annual.to_csv(OUTPUT_DIR / "_checkpoint_office_annual.csv", index=False)
for object_name in [
    "office_core", "office_quarter",
]:
    if object_name in globals():
        del globals()[object_name]
gc.collect()
print("Office row-level objects released; annual indicators retained.")

In [ ]:
eligible_msoa = msoa[msoa["MSOA21CD"].isin(eligible_codes)].copy()
retail_h2 = scan_openlocal_points(
    "RETAIL", eligible_msoa, ["MSOA21CD", "MSOA21NM"]
)
retail_h2["period"] = pd.to_datetime(retail_h2["period"])
retail_h2["year"] = retail_h2["period"].dt.year
retail_h2["total_floor_area"] = pd.to_numeric(
    retail_h2["total_floor_area"], errors="coerce"
)
retail_h2["rateable_value"] = pd.to_numeric(
    retail_h2["rateable_value"], errors="coerce"
)
retail_h2 = retail_h2.sort_values("period").drop_duplicates(
    ["period", "uarn"], keep="last"
)

def quarterly_retail_metrics(group):
    occupied = group["occupation_state"].astype("string").eq("OCCUPIED")
    vacant = group["occupation_state"].astype("string").eq("VACANT")
    classified = occupied | vacant
    return pd.Series({
        "ol_retail_units": group["uarn"].nunique(),
        "ol_retail_total_floor_area": pd.to_numeric(
            group["total_floor_area"], errors="coerce"
        ).sum(min_count=1),
        "ol_retail_total_rateable_value": pd.to_numeric(
            group["rateable_value"], errors="coerce"
        ).sum(min_count=1),
        "ol_retail_vacancy_proxy": (
            vacant.sum() / classified.sum() if classified.sum() else np.nan
        ),
    })

retail_quarter = (
    retail_h2.groupby(["period", "year", "MSOA21CD", "MSOA21NM"], observed=True)
    .apply(quarterly_retail_metrics, include_groups=False)
    .reset_index()
)
retail_annual = (
    retail_quarter.groupby(["year", "MSOA21CD", "MSOA21NM"], observed=True)
    .agg(
        retail_n_quarters=("period", "nunique"),
        **{
            col: (col, "mean")
            for col in retail_quarter.columns
            if col.startswith("ol_retail_")
        },
    )
    .reset_index()
)

retail_base_cols = [
    "ol_retail_units", "ol_retail_total_floor_area",
    "ol_retail_total_rateable_value", "ol_retail_vacancy_proxy",
]
retail_base = (
    retail_annual[retail_annual["year"].eq(2019)]
    [["MSOA21CD"] + retail_base_cols]
    .rename(columns={c: f"{c}_2019" for c in retail_base_cols})
)
retail_post = retail_annual[retail_annual["year"].isin(POST_YEARS)].merge(
    retail_base, on="MSOA21CD", how="inner", validate="many_to_one"
)
retail_post = retail_post[
    retail_post["ol_retail_units_2019"].ge(MIN_BASELINE_RETAIL_UNITS)
    & retail_post["retail_n_quarters"].ge(4)
].copy()
retail_post["ol_retail_unit_log_change"] = safe_log_ratio(
    retail_post["ol_retail_units"], retail_post["ol_retail_units_2019"]
)
retail_post["ol_retail_floor_log_change"] = safe_log_ratio(
    retail_post["ol_retail_total_floor_area"],
    retail_post["ol_retail_total_floor_area_2019"],
)
retail_post["ol_retail_value_log_change"] = safe_log_ratio(
    retail_post["ol_retail_total_rateable_value"],
    retail_post["ol_retail_total_rateable_value_2019"],
)
retail_post["ol_retail_vacancy_change"] = (
    retail_post["ol_retail_vacancy_proxy"]
    - retail_post["ol_retail_vacancy_proxy_2019"]
)

print("OpenLocal RQ2 retail MSOA-years:", len(retail_post))
print("OpenLocal RQ2 retail MSOAs:", retail_post["MSOA21CD"].nunique())

# Retain only annual indicators for the remaining analysis.
for object_name in [
    "retail_h2", "retail_quarter",
]:
    if object_name in globals():
        del globals()[object_name]
gc.collect()

## 5. Green Street retail outcomes

The historical Green Street panel is merged by MSOA and year. Event dates are
the dates on which Green Street detected the opening or closure and may lag
the underlying event by up to approximately six months. Annual aggregation is
therefore retained, and turnover outcomes are interpreted as recorded market
adjustment rather than exact event timing.

In [ ]:
gs = pd.read_csv(GS_POST_PANEL_PATH)
gs = gs[gs["year"].isin(POST_YEARS)].copy()
gs["gs_active_retail_2019"] = np.expm1(gs["log_active_retail_2019"])
gs = gs[gs["gs_active_retail_2019"].ge(MIN_BASELINE_GS_ACTIVE)].copy()
gs_keep = [
    "MSOA21CD", "year",
    "active_retail_log_change_2019",
    "log_active_retail_2019",
    "vacancy_share_change_2019",
    "vacancy_share_2019",
    "long_term_vacancy_share_change_2019",
    "long_term_vacancy_share_2019",
    "turnover_rate_change_2019",
    "turnover_rate_2019",
    "net_formation_rate_change_2019",
    "net_formation_rate_2019",
    "replacement_rate_change_2019",
    "replacement_rate_2019",
]
gs = gs[[c for c in gs_keep if c in gs.columns]]

h2_panel = h2_office_panel.merge(
    retail_post.drop(columns=["MSOA21NM"], errors="ignore"),
    on=["MSOA21CD", "year"],
    how="left",
    validate="one_to_one",
)
h2_panel = h2_panel.merge(
    gs,
    on=["MSOA21CD", "year"],
    how="left",
    validate="one_to_one",
)

print("Combined RQ2 panel rows:", len(h2_panel))
print("Office MSOAs:", h2_panel["MSOA21CD"].nunique())
print("Rows with OpenLocal retail:", h2_panel["ol_retail_units"].notna().sum())
print("Rows with Green Street:", h2_panel["active_retail_log_change_2019"].notna().sum())
display(
    h2_panel.groupby("study_submarket")
    .agg(
        office_MSOAs=("MSOA21CD", "nunique"),
        openlocal_rows=("ol_retail_units", "count"),
        greenstreet_rows=("active_retail_log_change_2019", "count"),
    )
)

## 6. Model specification

For each retail outcome, ordinary least squares estimates two complementary
specifications: one using the fragmentation index and one using the
persistent-large-office score. Both predictors are standardised, so each
coefficient represents the average outcome difference associated with a
one-standard-deviation increase in the relevant office structure.

The models control for:

- the 2019 level of the retail outcome;
- the 2019 scale of the local office market;
- concurrent change in total office floorspace;
- common differences between 2023, 2024 and 2025;
- persistent differences between the five office submarkets.

Standard errors are clustered by MSOA because each MSOA contributes three
post-pandemic observations. These models identify conditional associations,
not causal effects.

In [ ]:
outcome_specs = {
    "OpenLocal retail units": {
        "change": "ol_retail_unit_log_change",
        "baseline": "ol_retail_units_2019",
        "scale": 100,
        "family": "OpenLocal",
        "expected": "positive",
    },
    "OpenLocal rateable value": {
        "change": "ol_retail_value_log_change",
        "baseline": "ol_retail_total_rateable_value_2019",
        "scale": 100,
        "family": "OpenLocal",
        "expected": "positive",
    },
    "OpenLocal retail floor area": {
        "change": "ol_retail_floor_log_change",
        "baseline": "ol_retail_total_floor_area_2019",
        "scale": 100,
        "family": "OpenLocal",
        "expected": "positive",
    },
        "OpenLocal occupation-based vacancy proxy": {
        "change": "ol_retail_vacancy_change",
        "baseline": "ol_retail_vacancy_proxy_2019",
        "scale": 100,
        "family": "OpenLocal",
        "expected": "negative",
    },
    "Green Street active stock": {
        "change": "active_retail_log_change_2019",
        "baseline": "log_active_retail_2019",
        "scale": 100,
        "family": "Green Street",
        "expected": "positive",
    },
    "Green Street vacancy": {
        "change": "vacancy_share_change_2019",
        "baseline": "vacancy_share_2019",
        "scale": 100,
        "family": "Green Street",
        "expected": "negative",
    },
    "Green Street long-term vacancy": {
        "change": "long_term_vacancy_share_change_2019",
        "baseline": "long_term_vacancy_share_2019",
        "scale": 100,
        "family": "Green Street",
        "expected": "negative",
    },
    "Green Street turnover": {
        "change": "turnover_rate_change_2019",
        "baseline": "turnover_rate_2019",
        "scale": 100,
        "family": "Green Street",
        "expected": "contextual",
    },
    "Green Street net formation": {
        "change": "net_formation_rate_change_2019",
        "baseline": "net_formation_rate_2019",
        "scale": 100,
        "family": "Green Street",
        "expected": "positive",
    },
}

def fit_h2_model(data, label, spec, predictor="fragmentation_z"):
    needed = [
        "MSOA21CD", "year", "study_submarket", predictor,
        "office_floor_log_change", "log_office_floor_area_2019",
        spec["change"], spec["baseline"],
    ]
    sample = data[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
    sample["model_outcome"] = spec["scale"] * sample[spec["change"]]
    sample["baseline_outcome"] = sample[spec["baseline"]]
    formula = (
        "model_outcome ~ " + predictor
        + " + office_floor_log_change + log_office_floor_area_2019"
        + " + baseline_outcome + C(year) + C(study_submarket)"
    )
    model = smf.ols(formula, data=sample).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["MSOA21CD"]},
    )
    ci = model.conf_int().loc[predictor]
    row = {
        "outcome": label,
        "family": spec["family"],
        "predictor": predictor,
        "n_obs": int(model.nobs),
        "n_msoa": sample["MSOA21CD"].nunique(),
        "beta": model.params[predictor],
        "robust_se": model.bse[predictor],
        "ci_low": ci.iloc[0],
        "ci_high": ci.iloc[1],
        "p_value": model.pvalues[predictor],
        "r_squared": model.rsquared,
        "expected_direction": spec["expected"],
    }
    return row, model, sample

main_predictors = {
    "fragmentation_z": "Office fragmentation",
    "persistent_large_z": "Persistent large-office stock",
}
model_rows = []
model_objects = {}
model_samples = {}
for predictor, predictor_label in main_predictors.items():
    for label, spec in outcome_specs.items():
        if spec["change"] not in h2_panel or spec["baseline"] not in h2_panel:
            continue
        row, model, sample = fit_h2_model(
            h2_panel, label, spec, predictor=predictor
        )
        row["predictor_label"] = predictor_label
        model_rows.append(row)
        model_objects[(label, predictor_label)] = model
        model_samples[(label, predictor_label)] = sample

h2_models = pd.DataFrame(model_rows)
display(h2_models)

## 6.1 Supplementary office-occupation diagnostic

The fragmentation and persistent-large-office specifications test **office-stock
structure**. They do not by themselves show whether offices are more or less
occupied. OpenLocal therefore provides a separate office occupation-based
vacancy proxy, calculated from official VOA occupation states as the share of
classified office hereditaments recorded vacant.

This diagnostic estimates whether an increase in the office vacancy proxy is
associated with retail change after accounting for the 2019 office vacancy
level, baseline office floorspace, concurrent office-floorspace change, the
retail baseline, year and submarket. It does not observe physical attendance or
desk utilisation. A non-significant estimate should therefore be interpreted as
no clear association in the recorded property-occupation data, not evidence
that offices were fully used.

In [ ]:
def fit_office_occupation_model(data, label, spec):
    predictor = "office_vacancy_proxy_change"
    needed = [
        "MSOA21CD", "year", "study_submarket", predictor,
        "office_vacancy_proxy_2019", "office_floor_log_change",
        "log_office_floor_area_2019", spec["change"], spec["baseline"],
    ]
    sample = data[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(sample) < 30:
        return None, None, None
    sample["model_outcome"] = spec["scale"] * sample[spec["change"]]
    sample["baseline_outcome"] = sample[spec["baseline"]]
    formula = (
        "model_outcome ~ office_vacancy_proxy_change"
        " + office_vacancy_proxy_2019 + office_floor_log_change"
        " + log_office_floor_area_2019 + baseline_outcome"
        " + C(year) + C(study_submarket)"
    )
    model = smf.ols(formula, data=sample).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["MSOA21CD"]},
    )
    ci = model.conf_int().loc["office_vacancy_proxy_change"]
    return {
        "outcome": label,
        "family": spec["family"],
        "n_obs": int(model.nobs),
        "n_msoa": sample["MSOA21CD"].nunique(),
        "beta_per_10pp_office_vacancy": 0.10 * model.params["office_vacancy_proxy_change"],
        "ci_low_per_10pp": 0.10 * ci.iloc[0],
        "ci_high_per_10pp": 0.10 * ci.iloc[1],
        "p_value": model.pvalues["office_vacancy_proxy_change"],
        "r_squared": model.rsquared,
    }, model, sample

occupation_rows = []
occupation_models = {}
occupation_samples = {}
for label, spec in outcome_specs.items():
    row, model, sample = fit_office_occupation_model(h2_panel, label, spec)
    if row is not None:
        occupation_rows.append(row)
        occupation_models[label] = model
        occupation_samples[label] = sample

office_occupation_results = pd.DataFrame(occupation_rows)
display(office_occupation_results)

## 6.2 Figure: adjusted office occupation and retail outcomes

This figure translates the supplementary model into three adjusted predicted
relationships. It asks: **holding the stated retail and office conditions
constant, how are selected retail outcomes expected to differ as the recorded
office vacancy proxy changes?** The three outcomes are deliberately selected to
show one property-occupation result, one retail-reallocation result, and one
null result for active consumer-facing retail stock. This is easier to read than
a full coefficient list; the complete nine-outcome summary is retained as an
appendix figure.

In [ ]:
def average_adjusted_office_occupation_prediction(model, sample, points=80):
    predictor = "office_vacancy_proxy_change"
    lo, hi = sample[predictor].quantile([0.02, 0.98])
    grid = np.linspace(lo, hi, points)
    params = model.params.to_numpy()
    covariance = model.cov_params().to_numpy()
    design_info = model.model.data.design_info
    rows = []
    for value in grid:
        new_data = sample.copy()
        new_data[predictor] = value
        design = np.asarray(
            build_design_matrices([design_info], new_data)[0], dtype=float
        )
        average_design = design.mean(axis=0)
        prediction = float(average_design @ params)
        variance = float(average_design @ covariance @ average_design)
        standard_error = np.sqrt(max(variance, 0))
        rows.append({
            "office_vacancy_change": value,
            "prediction": prediction,
            "ci_low": prediction - 1.96 * standard_error,
            "ci_high": prediction + 1.96 * standard_error,
        })
    return pd.DataFrame(rows)

condition_panels = [
    (
        "OpenLocal occupation-based vacancy proxy",
        "A. Retail property occupation",
        "Change in OpenLocal retail occupation-based vacancy (pp)",
        PALETTE["blue_dark"],
    ),
    (
        "Green Street net formation",
        "B. Recorded retail reallocation",
        "Change in Green Street net formation (pp)",
        PALETTE["orange_dark"],
    ),
    (
        "Green Street active stock",
        "C. Active consumer-facing retail stock",
        "Change in Green Street active stock (%)",
        PALETTE["green"],
    ),
]

fig, axes = plt.subplots(1, 3, figsize=(13.3, 5.4), constrained_layout=True)
for ax, (outcome, title, ylabel, color) in zip(axes, condition_panels):
    prediction = average_adjusted_office_occupation_prediction(
        occupation_models[outcome], occupation_samples[outcome]
    )
    ax.fill_between(
        prediction["office_vacancy_change"] * 100,
        prediction["ci_low"], prediction["ci_high"],
        color=color, alpha=0.18, linewidth=0,
    )
    ax.plot(
        prediction["office_vacancy_change"] * 100,
        prediction["prediction"], color=color, linewidth=3,
    )
    ax.axvline(0, color="#4b5563", linestyle="--", linewidth=1.1)
    ax.set_title(title, loc="left", fontsize=11.3)
    ax.set_xlabel("Change in recorded office vacancy (percentage points)", fontsize=8.8)
    ax.set_ylabel(ylabel, fontsize=8.8)
    ax.grid(axis="x", alpha=0.18)
    sns.despine(ax=ax)

fig.suptitle(
    "RQ2 supplementary diagnostic: adjusted retail outcomes as recorded office vacancy changes",
    x=0.01, ha="left", fontsize=16.5,
)
fig.savefig(FIGURE_DIR / "fig_04_h2_office_occupation_diagnostic.png", dpi=300, bbox_inches="tight")
plt.show()

# Full diagnostic retained for the dissertation appendix.
appendix_panels = [
    ("Retail-stock outcomes (% change)", ["OpenLocal retail units", "OpenLocal rateable value", "OpenLocal retail floor area", "Green Street active stock"]),
    ("Retail-vacancy outcomes (percentage-point change)", ["OpenLocal occupation-based vacancy proxy", "Green Street vacancy", "Green Street long-term vacancy"]),
    ("Retail-flow outcomes (percentage-point change)", ["Green Street turnover", "Green Street net formation"]),
]
appendix_short_labels = {
    "OpenLocal retail units": "Retail units (OL)",
    "OpenLocal rateable value": "Rateable value (OL)",
    "OpenLocal retail floor area": "Retail floor area (OL)",
    "Green Street active stock": "Active retail stock (GS)",
    "OpenLocal occupation-based vacancy proxy": "Occupation vacancy (OL)",
    "Green Street vacancy": "Observed vacancy (GS)",
    "Green Street long-term vacancy": "Long-term vacancy (GS)",
    "Green Street turnover": "Recorded turnover (GS)",
    "Green Street net formation": "Recorded net formation (GS)",
}
fig, axes = plt.subplots(1, 3, figsize=(13.3, 5.4), constrained_layout=True)
for ax, (title, labels) in zip(axes, appendix_panels):
    subset = office_occupation_results.set_index("outcome").reindex(labels).reset_index()
    y = np.arange(len(subset))[::-1]
    ax.errorbar(
        subset["beta_per_10pp_office_vacancy"], y,
        xerr=[
            subset["beta_per_10pp_office_vacancy"] - subset["ci_low_per_10pp"],
            subset["ci_high_per_10pp"] - subset["beta_per_10pp_office_vacancy"],
        ],
        fmt="o", color=PALETTE["orange_dark"], ecolor="#b7c3cc",
        elinewidth=2.0, capsize=3.5, markersize=6.5,
    )
    ax.axvline(0, color="#4b5563", linestyle="--", linewidth=1.1)
    ax.set_yticks(y, [appendix_short_labels[label] for label in labels], fontsize=8.7)
    ax.set_title(title, loc="left", fontsize=11.3)
    ax.set_xlabel("Estimated change for a 10 pp rise in office vacancy", fontsize=8.8)
    ax.grid(axis="x", alpha=0.18)
fig.suptitle(
    "Appendix: full office-occupation diagnostic",
    x=0.01, ha="left", fontsize=16.5,
)
fig.savefig(
    FIGURE_DIR / "fig_a2_h2_office_occupation_coefficients.png",
    dpi=300, bbox_inches="tight",
)
plt.show()

## 7. Figure 1: spatial pattern of office-stock adjustment

The two maps show the mean 2023-2025 values of the office-fragmentation index
and the persistent-large-office score. Mapping both measures is important
because they represent distinct adjustment pathways rather than opposite ends
of one scale. The maps are descriptive: the controlled models, rather than the
colour pattern alone, answer RQ2.

In [ ]:
fragment_mean = (
    h2_panel.groupby(["MSOA21CD", "MSOA21NM", "study_submarket"], observed=True)
    .agg(
        fragmentation_z=("fragmentation_z", "mean"),
        persistent_large_z=("persistent_large_z", "mean"),
        office_floor_log_change=("office_floor_log_change", "mean"),
    )
    .reset_index()
)
h2_map = msoa.merge(fragment_mean, on=["MSOA21CD", "MSOA21NM"], how="inner")
h2_map_27700 = h2_map.to_crs("EPSG:27700")
core_27700 = core_markets.to_crs("EPSG:27700")
# The source layer contains many small market polygons.  Dissolving them here
# makes the five analytical office-submarket boundaries legible on the maps.
core_submarket_boundaries = core_27700.dissolve(by="study_submarket", as_index=False)
lad_27700 = london_lad.to_crs("EPSG:27700")

cxmin, cymin, cxmax, cymax = core_27700.total_bounds
map_pad = 2500
fig, axes = plt.subplots(2, 1, figsize=(11.2, 11.8), constrained_layout=True)
map_specs = [
    (
        "fragmentation_z",
        "A. Office fragmentation",
        "Higher values indicate more units, smaller median units and a shift towards small offices",
        "Blues",
        "Lower fragmentation  ←  relative score  →  Higher fragmentation",
    ),
    (
        "persistent_large_z",
        "B. Persistent large-office structure",
        "Higher values indicate a large baseline floorspace share that changed comparatively little",
        "Oranges",
        "Lower persistence  ←  relative score  →  Higher persistence",
    ),
]
for ax, (column, title, subtitle, colour_map, colourbar_label) in zip(axes, map_specs):
    values = h2_map_27700[column].dropna()
    vmin = float(values.quantile(0.02))
    vmax = float(values.quantile(0.98))
    norm = Normalize(vmin=vmin, vmax=vmax, clip=True)
    lad_27700.boundary.plot(ax=ax, color="#c5c8cc", linewidth=0.42)
    h2_map_27700.plot(
        ax=ax,
        column=column,
        cmap=colour_map,
        norm=norm,
        edgecolor="white",
        linewidth=0.34,
    )
    core_submarket_boundaries.boundary.plot(ax=ax, color="#30363d", linewidth=1.35, zorder=5)
    ax.set_xlim(cxmin - map_pad, cxmax + map_pad)
    ax.set_ylim(cymin - map_pad, cymax + map_pad)
    ax.set_axis_off()
    ax.set_title(title, loc="left", fontsize=18, pad=20, fontweight="bold")
    ax.text(
        0.0, 1.01, subtitle,
        transform=ax.transAxes, ha="left", va="bottom",
        fontsize=11.5, color="#5b6169",
    )
    sm = plt.cm.ScalarMappable(cmap=colour_map, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(
        sm, ax=ax, orientation="horizontal",
        fraction=0.045, pad=0.015, shrink=0.68, aspect=34,
    )
    cbar.set_label(colourbar_label, fontsize=12)
    cbar.ax.tick_params(labelsize=10.5)

fig.suptitle(
    "RQ2 Spatial Distribution of Office-Stock Adjustment",
    x=0.01, ha="left", fontsize=23, y=1.02,
)
fig.savefig(FIGURE_DIR / "fig_01_h2_office_fragmentation_spatial_pattern.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Figure 2: adjusted relationships

These panels translate the controlled regressions into predicted retail
changes across the observed range of each office measure. The prediction at
each point averages over the observed baseline retail condition, office-market
scale, office-floorspace change, year and submarket composition. The shaded
band is the 95% confidence interval. Showing both predictors for both outcomes
avoids selecting only the statistically significant relationships.

In [ ]:
def average_adjusted_prediction(model, sample, predictor, points=80):
    lo, hi = sample[predictor].quantile([0.02, 0.98])
    grid = np.linspace(lo, hi, points)
    params = model.params.to_numpy()
    covariance = model.cov_params().to_numpy()
    design_info = model.model.data.design_info
    rows = []
    for value in grid:
        new_data = sample.copy()
        new_data[predictor] = value
        design = np.asarray(
            build_design_matrices([design_info], new_data)[0],
            dtype=float,
        )
        average_design = design.mean(axis=0)
        prediction = float(average_design @ params)
        variance = float(average_design @ covariance @ average_design)
        standard_error = np.sqrt(max(variance, 0))
        rows.append({
            "predictor_value": value,
            "prediction": prediction,
            "ci_low": prediction - 1.96 * standard_error,
            "ci_high": prediction + 1.96 * standard_error,
        })
    return pd.DataFrame(rows)

adjusted_specs = [
    (
        "Green Street active stock",
        "Office fragmentation",
        "A. Fragmentation and active retail stock",
        "Change in active retail stock relative to 2019 (%)",
    ),
    (
        "Green Street active stock",
        "Persistent large-office stock",
        "B. Persistent large offices and active retail stock",
        "Change in active retail stock relative to 2019 (%)",
    ),
    (
        "Green Street net formation",
        "Office fragmentation",
        "C. Fragmentation and recorded net formation",
        "Change in recorded net formation relative to 2019 (pp)",
    ),
    (
        "Green Street net formation",
        "Persistent large-office stock",
        "D. Persistent large offices and recorded net formation",
        "Change in recorded net formation relative to 2019 (pp)",
    ),
]

fig, axes = plt.subplots(2, 2, figsize=(12.2, 9.4), constrained_layout=True)
for ax, (outcome, predictor_label, title, y_label) in zip(axes.flat, adjusted_specs):
    predictor = next(
        key for key, label in main_predictors.items()
        if label == predictor_label
    )
    model = model_objects[(outcome, predictor_label)]
    sample = model_samples[(outcome, predictor_label)]
    prediction = average_adjusted_prediction(model, sample, predictor)
    model_row = h2_models[
        h2_models["outcome"].eq(outcome)
        & h2_models["predictor_label"].eq(predictor_label)
    ].iloc[0]
    color = (
        PALETTE["blue_dark"]
        if predictor_label == "Office fragmentation"
        else PALETTE["orange_dark"]
    )
    ax.fill_between(
        prediction["predictor_value"],
        prediction["ci_low"],
        prediction["ci_high"],
        color=color,
        alpha=0.17,
        linewidth=0,
    )
    ax.plot(
        prediction["predictor_value"],
        prediction["prediction"],
        color=color,
        linewidth=3,
    )
    ax.axhline(0, color="#6b7280", linewidth=1.1, linestyle="--")
    ax.set_title(title, loc="left", fontsize=15, fontweight="bold")
    ax.set_xlabel("Office measure (standard deviations)", fontsize=11.5)
    ax.set_ylabel(y_label, fontsize=11.5)
    ax.text(
        0.03, 0.95,
        f"beta={model_row['beta']:.2f}; 95% CI "
        f"[{model_row['ci_low']:.2f}, {model_row['ci_high']:.2f}]; "
        f"p={model_row['p_value']:.3f}",
        transform=ax.transAxes, ha="left", va="top",
        fontsize=10.5,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 3},
    )
    sns.despine(ax=ax)

fig.suptitle(
    "RQ2 Adjusted Green Street Retail Relationships",
    fontsize=22, x=0.01, ha="left", y=1.02,
)
fig.savefig(FIGURE_DIR / "fig_02_h2_adjusted_relationships.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Figure 3: controlled RQ2 coefficient summary

The coefficient summary separates outcomes with different units so that the
large active-stock estimate does not visually compress vacancy, turnover and
formation rates. Each point is the estimated outcome difference associated
with a one-standard-deviation increase in the office measure. Horizontal lines
are 95% confidence intervals and the grey line marks no estimated association.

In [ ]:
coefficient_panels = [
    (
        "A. Retail-stock outcomes (% change)",
        [
            "OpenLocal retail units",
            "OpenLocal rateable value",
            "OpenLocal retail floor area",
            "Green Street active stock",
        ],
        "Estimated percentage change",
    ),
    (
        "B. Vacancy outcomes (percentage-point change)",
        [
        "OpenLocal occupation-based vacancy proxy",
            "Green Street vacancy",
            "Green Street long-term vacancy",
        ],
        "Estimated percentage-point change",
    ),
    (
        "C. Retail-flow outcomes (percentage-point change)",
        [
            "Green Street turnover",
            "Green Street net formation",
        ],
        "Estimated percentage-point change",
    ),
]
short_labels = {
    "OpenLocal retail units": "Retail units (OL)",
    "OpenLocal rateable value": "Rateable value (OL)",
    "OpenLocal retail floor area": "Retail floor area (OL)",
    "Green Street active stock": "Active consumer-facing stock (GS)",
        "OpenLocal occupation-based vacancy proxy": "Occupation-based vacancy (OL)",
    "Green Street vacancy": "Observed vacancy (GS)",
    "Green Street long-term vacancy": "Long-term vacancy (GS)",
    "Green Street turnover": "Recorded turnover (GS)",
    "Green Street net formation": "Recorded net formation (GS)",
}
styles = [
    ("Office fragmentation", PALETTE["blue_dark"], -0.13, "o"),
    ("Persistent large-office stock", PALETTE["orange_dark"], 0.13, "s"),
]

fig, axes = plt.subplots(
    3, 1, figsize=(11.2, 12.5),
    gridspec_kw={"height_ratios": [1.25, 1.0, 0.85]},
    constrained_layout=True,
)
for ax, (title, outcome_order, x_label) in zip(axes, coefficient_panels):
    display_order = outcome_order[::-1]
    y = np.arange(len(display_order))
    for predictor_label, predictor_color, offset, marker in styles:
        plot = (
            h2_models[
                h2_models["predictor_label"].eq(predictor_label)
                & h2_models["outcome"].isin(display_order)
            ]
            .set_index("outcome")
            .reindex(display_order)
            .reset_index()
        )
        ax.errorbar(
            plot["beta"], y + offset,
            xerr=[plot["beta"] - plot["ci_low"], plot["ci_high"] - plot["beta"]],
            fmt=marker, color=predictor_color, ecolor=predictor_color,
            elinewidth=2.2, capsize=4, markersize=8,
            label=predictor_label,
        )
        for yi, row in plot.iterrows():
            if row["p_value"] < 0.05:
                direction = 1 if row["beta"] >= 0 else -1
                anchor = row["ci_high"] if direction > 0 else row["ci_low"]
                ax.annotate(
                    f"beta={row['beta']:.2f}, p={row['p_value']:.3f}",
                    xy=(anchor, yi + offset),
                    xytext=(8 * direction, 0),
                    textcoords="offset points",
                    ha="left" if direction > 0 else "right",
                    va="center",
                    color=predictor_color,
                    fontsize=9.5,
                    fontweight="bold",
                )
    ax.axvline(0, color="#60656f", linewidth=1.2, linestyle="--")
    ax.set_yticks(y)
    ax.set_yticklabels([short_labels[item] for item in display_order], fontsize=11)
    ax.set_title(title, loc="left", fontsize=16, fontweight="bold")
    ax.set_xlabel(x_label, fontsize=11.5)
    ax.grid(axis="x", alpha=0.18)
    sns.despine(ax=ax)

legend_handles = [
    plt.Line2D(
        [0], [0], marker="o", linestyle="none",
        markerfacecolor=PALETTE["blue_dark"],
        markeredgecolor=PALETTE["blue_dark"],
        label="Office fragmentation",
    ),
    plt.Line2D(
        [0], [0], marker="s", linestyle="none",
        markerfacecolor=PALETTE["orange_dark"],
        markeredgecolor=PALETTE["orange_dark"],
        label="Persistent large-office stock",
    ),
]
fig.legend(
    handles=legend_handles,
    frameon=False, loc="upper right",
    bbox_to_anchor=(0.99, 1.035), fontsize=10.5, ncol=2,
)
fig.suptitle(
    "RQ2 Controlled Model Coefficients",
    fontsize=23, x=0.01, ha="left", y=1.035,
)
fig.savefig(FIGURE_DIR / "fig_03_h2_controlled_coefficients.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Component and spatial sensitivity

The index is unpacked by re-estimating the same models with each component in
turn. This reveals whether a result depends on unit-count growth, falling
median size, growth in small units, or decline in large units. Moran's I then
tests whether nearby MSOAs retain similar model residuals. Positive residual
clustering would indicate that the non-spatial regression has not captured all
geographical structure and should be treated cautiously.

The main analysis requires at least 20 baseline office and retail units. A
separate threshold sensitivity repeats the controlled models at 30 and 50
units. This tests whether the findings depend on unstable changes in small
local markets; it does not redefine the main study sample.

In [ ]:
component_labels = {
    "z_office_unit_log_change": "More office units",
    "z_office_median_size_log_change": "Smaller median unit",
    "z_small_unit_share_change": "Larger small-unit share",
    "z_large_unit_share_change": "Smaller large-unit share",
}
component_rows = []
for predictor, predictor_label in component_labels.items():
    for label, spec in outcome_specs.items():
        if spec["change"] not in h2_panel or spec["baseline"] not in h2_panel:
            continue
        row, _, _ = fit_h2_model(h2_panel, label, spec, predictor=predictor)
        row["component_label"] = predictor_label
        component_rows.append(row)
component_models = pd.DataFrame(component_rows)

heat = component_models.pivot(
    index="outcome", columns="component_label", values="beta"
)
heat = heat.reindex([x for x in outcome_specs if x in heat.index])
fig, ax = plt.subplots(figsize=(11.5, 8))
vmax = np.nanquantile(np.abs(heat.to_numpy()), 0.95)
sns.heatmap(
    heat, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
    annot=True, fmt=".1f", linewidths=0.5, cbar_kws={"label": "Coefficient"},
    ax=ax,
)
ax.set_title("RQ2 Sensitivity to Individual Fragmentation Components", loc="left", fontsize=21)
ax.set_xlabel("")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_05_h2_component_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Repeat the controlled RQ2 models at progressively stricter baseline-stock
# thresholds. The combined panel already satisfies the main 20-unit rule, so
# the 30- and 50-unit samples can be formed without re-reading row-level data.
threshold_rows = []
for threshold in [20, 30, 50]:
    for predictor, predictor_label in main_predictors.items():
        for label, spec in outcome_specs.items():
            if spec["change"] not in h2_panel or spec["baseline"] not in h2_panel:
                continue
            threshold_sample = h2_panel[
                h2_panel["office_units_2019"].ge(threshold)
            ].copy()
            if spec["family"] == "OpenLocal":
                threshold_sample = threshold_sample[
                    threshold_sample["ol_retail_units_2019"].ge(threshold)
                ].copy()
            else:
                threshold_sample["gs_active_retail_2019"] = np.expm1(
                    threshold_sample["log_active_retail_2019"]
                )
                threshold_sample = threshold_sample[
                    threshold_sample["gs_active_retail_2019"].ge(threshold)
                ].copy()
            row, _, _ = fit_h2_model(
                threshold_sample, label, spec, predictor=predictor
            )
            row.update({
                "minimum_2019_units": threshold,
                "predictor_label": predictor_label,
            })
            threshold_rows.append(row)

threshold_sensitivity = pd.DataFrame(threshold_rows)

# The appendix figure focuses on the outcomes that most directly describe
# retail stock and replacement. The complete set is retained in the CSV.
figure_outcomes = [
    "OpenLocal retail units",
    "OpenLocal rateable value",
    "Green Street active stock",
    "Green Street net formation",
]
threshold_colors = {
    20: "#6f9b88",
    30: "#d98745",
    50: "#174a74",
}
fig, axes = plt.subplots(1, 2, figsize=(11.2, 6.0), sharey=True)
for ax, predictor_label in zip(axes, main_predictors.values()):
    subset = threshold_sensitivity[
        threshold_sensitivity["predictor_label"].eq(predictor_label)
        & threshold_sensitivity["outcome"].isin(figure_outcomes)
    ]
    for outcome_y, outcome_label in enumerate(figure_outcomes):
        for offset, threshold in zip([-0.18, 0, 0.18], [20, 30, 50]):
            row = subset[
                subset["outcome"].eq(outcome_label)
                & subset["minimum_2019_units"].eq(threshold)
            ]
            if row.empty:
                continue
            row = row.iloc[0]
            ax.errorbar(
                row["beta"], outcome_y + offset,
                xerr=[[row["beta"] - row["ci_low"]], [row["ci_high"] - row["beta"]]],
                fmt="o",
                color=threshold_colors[threshold],
                capsize=3,
                markersize=6,
            )
    ax.axvline(0, color="#6b7280", linewidth=1.2, linestyle="--")
    ax.set_title(predictor_label, fontsize=14)
    ax.set_xlabel("Controlled coefficient (95% CI)", fontsize=11)
    ax.grid(axis="x", alpha=0.18)
axes[0].set_yticks(range(len(figure_outcomes)), figure_outcomes, fontsize=11)
handles = [
    plt.Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=threshold_colors[t],
        markeredgecolor=threshold_colors[t],
        label=f"At least {t} units",
    )
    for t in [20, 30, 50]
]
fig.legend(
    handles=handles, loc="lower center", ncol=3,
    frameon=False, fontsize=10, bbox_to_anchor=(0.5, 0.01),
)
fig.suptitle(
    "RQ2 Sensitivity to Minimum 2019 Office and Retail Stock",
    x=0.02, ha="left", fontsize=18,
)
fig.tight_layout(rect=[0, 0.10, 1, 0.93])
fig.savefig(
    FIGURE_DIR / "fig_06_rq2_baseline_stock_threshold_sensitivity.png",
    dpi=300, bbox_inches="tight",
)
plt.show()
display(threshold_sensitivity[[
    "minimum_2019_units", "predictor_label", "outcome",
    "n_obs", "n_msoa", "beta", "ci_low", "ci_high", "p_value",
]])

In [ ]:
def moran_i_permutation(gdf, values, permutations=999, seed=42):
    work = gdf[["MSOA21CD", "geometry"]].copy()
    work["value"] = values
    work = work.dropna(subset=["value"]).reset_index(drop=True)
    n = len(work)
    if n < 4:
        return np.nan, np.nan, n
    w = np.zeros((n, n), dtype=float)
    geoms = work.geometry.to_list()
    for i in range(n):
        for j in range(i + 1, n):
            if geoms[i].touches(geoms[j]):
                w[i, j] = 1
                w[j, i] = 1
    row_sums = w.sum(axis=1)
    keep = row_sums > 0
    w = w[np.ix_(keep, keep)]
    x = work.loc[keep, "value"].to_numpy(dtype=float)
    n = len(x)
    row_sums = w.sum(axis=1)
    w = w / row_sums[:, None]
    z = x - x.mean()
    denom = np.sum(z ** 2)
    observed = (n / w.sum()) * (np.sum(w * np.outer(z, z)) / denom)
    rng = np.random.default_rng(seed)
    sims = np.empty(permutations)
    for p in range(permutations):
        zp = rng.permutation(z)
        sims[p] = (n / w.sum()) * (np.sum(w * np.outer(zp, zp)) / denom)
    p_value = (np.sum(np.abs(sims) >= abs(observed)) + 1) / (permutations + 1)
    return observed, p_value, n

model_moran_rows = []
geometry_lookup = msoa[["MSOA21CD", "geometry"]]
for (label, predictor_label), model in model_objects.items():
    sample = model_samples[(label, predictor_label)].copy()
    sample["residual"] = model.resid
    mean_resid = sample.groupby("MSOA21CD", as_index=False)["residual"].mean()
    residual_map = geometry_lookup.merge(mean_resid, on="MSOA21CD", how="inner")
    moran_i, moran_p, n = moran_i_permutation(
        residual_map, residual_map["residual"], permutations=999
    )
    model_moran_rows.append({
        "outcome": label,
        "predictor": predictor_label,
        "moran_i": moran_i,
        "permutation_p": moran_p,
        "n_msoa": n,
    })
moran_results = pd.DataFrame(model_moran_rows)
display(moran_results)

fig, ax = plt.subplots(figsize=(10.8, 8.6))
moran_plot = moran_results.copy()
moran_plot["short_outcome"] = moran_plot["outcome"].map(short_labels)
outcome_order = [
    short_labels[item]
    for item in outcome_specs
    if item in set(moran_plot["outcome"])
][::-1]
y = np.arange(len(outcome_order))
for predictor_label, predictor_color, offset, marker in styles:
    plot = (
        moran_plot[moran_plot["predictor"].eq(predictor_label)]
        .set_index("short_outcome")
        .reindex(outcome_order)
        .reset_index()
    )
    significant = plot["permutation_p"].lt(0.05)
    ax.scatter(
        plot.loc[~significant, "moran_i"], y[~significant] + offset,
        marker=marker, s=58, facecolor="white",
        edgecolor=predictor_color, linewidth=1.8,
        label=predictor_label if predictor_label == styles[0][0] else None,
    )
    ax.scatter(
        plot.loc[significant, "moran_i"], y[significant] + offset,
        marker=marker, s=72, facecolor=predictor_color,
        edgecolor=predictor_color, linewidth=1.3,
        label=(
            predictor_label
            if predictor_label != styles[0][0]
            else None
        ),
    )
    for yi, row in plot.iterrows():
        if row["permutation_p"] < 0.05:
            ax.annotate(
                f"p={row['permutation_p']:.3f}",
                (row["moran_i"], yi + offset),
                xytext=(7, 0), textcoords="offset points",
                va="center", fontsize=9.5, color=predictor_color,
            )
ax.axvline(0, color="#60656f", linewidth=1.2, linestyle="--")
ax.set_yticks(y)
ax.set_yticklabels(outcome_order, fontsize=10.5)
ax.set_xlabel("Moran's I of mean MSOA model residuals", fontsize=12)
fig.suptitle(
    "RQ2 Residual Spatial Autocorrelation Diagnostic",
    x=0.01, ha="left", fontsize=20, fontweight="bold", y=0.995,
)
fig.text(
    0.08, 0.955,
    "Filled markers indicate permutation p < 0.05; positive values indicate clustering and negative values spatial dispersion.",
    ha="left", va="top", fontsize=10.5, color="#5b6169",
)
legend_handles = [
    plt.Line2D(
        [0], [0], marker="o", linestyle="none",
        markerfacecolor="white", markeredgecolor=PALETTE["blue_dark"],
        markeredgewidth=1.8, label="Office fragmentation",
    ),
    plt.Line2D(
        [0], [0], marker="s", linestyle="none",
        markerfacecolor="white", markeredgecolor=PALETTE["orange_dark"],
        markeredgewidth=1.8, label="Persistent large-office stock",
    ),
]
fig.legend(
    handles=legend_handles, frameon=False,
    loc="lower center", bbox_to_anchor=(0.5, 0.01),
    fontsize=10, ncol=2,
)
ax.grid(axis="x", alpha=0.18)
sns.despine(ax=ax)
fig.tight_layout(rect=[0, 0.06, 1, 0.93])
fig.savefig(
    FIGURE_DIR / "fig_04_h2_moran_residual_diagnostics.png",
    dpi=300, bbox_inches="tight",
)
plt.show()

## 11. Export analysis-ready outputs

Restricted row-level data remain outside the public repository. These derived
MSOA-year indicators, model summaries and publication figures are exported for
dissertation drafting. A public replication repository should contain code,
field descriptions and instructions for obtaining OpenLocal, but not the
commercial Green Street source files.

In [ ]:
exports = {
    "h2_office_msoa_year_indicators.csv": office_annual,
    "h2_office_fragmentation_panel.csv": h2_office_panel,
    "h2_combined_analysis_panel.csv": h2_panel,
    "h2_main_model_results.csv": h2_models,
    "h2_office_occupation_diagnostic_results.csv": office_occupation_results,
    "h2_component_sensitivity_results.csv": component_models,
    "rq2_baseline_stock_threshold_sensitivity.csv": threshold_sensitivity,
    "h2_moran_residual_diagnostics.csv": moran_results,
}
for filename, dataframe in exports.items():
    dataframe.to_csv(OUTPUT_DIR / filename, index=False)

figure_manifest = pd.DataFrame([
    {
        "figure": "Figure 0",
        "file": "fig_00_h2_office_market_adjustment_components.png",
        "role": "Joint diagnostic of office unit, size, floorspace, value and occupation changes",
        "placement": "RQ2 results",
    },
    {
        "figure": "Figure 1",
        "file": "fig_01_h2_office_fragmentation_spatial_pattern.png",
        "role": "Spatial distribution of fragmentation and persistent large-office structure",
        "placement": "RQ2 results",
    },
    {
        "figure": "Figure 2",
        "file": "fig_02_h2_adjusted_relationships.png",
        "role": "Controlled adjusted relationships for active stock and net formation",
        "placement": "RQ2 results",
    },
    {
        "figure": "Figure 3",
        "file": "fig_03_h2_controlled_coefficients.png",
        "role": "Primary controlled results separated by outcome units",
        "placement": "RQ2 appendix",
    },
    {
        "figure": "Figure 4",
        "file": "fig_04_h2_office_occupation_diagnostic.png",
        "role": "Adjusted relationship between recorded office vacancy change and selected retail outcomes",
        "placement": "RQ2 results",
    },
    {
        "figure": "Appendix Figure A2",
        "file": "fig_a2_h2_office_occupation_coefficients.png",
        "role": "Full office-occupation diagnostic across all retail outcomes",
        "placement": "RQ2 appendix",
    },
    {
        "figure": "Figure 5",
        "file": "fig_04_h2_moran_residual_diagnostics.png",
        "role": "Residual spatial-autocorrelation diagnostic",
        "placement": "RQ2 appendix",
    },
    {
        "figure": "Figure 6",
        "file": "fig_05_h2_component_sensitivity.png",
        "role": "Checks which fragmentation components drive results",
        "placement": "Appendix",
    },
    {
        "figure": "Figure 7",
        "file": "fig_06_rq2_baseline_stock_threshold_sensitivity.png",
        "role": "Checks sensitivity to minimum baseline office and retail stock",
        "placement": "Appendix",
    },
])
figure_manifest.to_csv(OUTPUT_DIR / "figure_manifest.csv", index=False)

manifest = pd.DataFrame([
    {"file": name, "rows": len(df), "columns": len(df.columns)}
    for name, df in exports.items()
])
display(manifest)
display(figure_manifest)

## 12. Reading the results

The answer to RQ2 is based on the **pattern across outcome families**, not on a
single p-value. The first analytical expectation would be consistent with
greater fragmentation being associated with growth in retail stock or value,
lower vacancy or long-term vacancy, and positive net formation after
office-market scale and submarket differences are accounted for. Turnover is
interpreted contextually: a moderate increase may indicate active adaptation,
whereas turnover without net formation or alongside persistent vacancy may
indicate instability.

If OpenLocal and Green Street differ, the result should be reported as a
difference between administrative property stock and visible consumer-facing
activity, consistent with the source-reconciliation findings.